In [ ]:
# =================================================================
# SOTA ISLES-2022: Final P100 DynUNet (nnU-Net) Engine
# - Architecture: DynUNet (ISLES-2022 Winning Architecture)
# - Hardware Profile: Tesla P100 (Pure FP32 Precision)
# - BUG FIX: dropout=None (Bypasses the missing PyTorch 3D kernel)
# - Pipeline: Foreground Oversampling (RandCropByPosNegLabeld)
# - Collation: Crash-Proof list_data_collate
# - FORCES 100 epochs (Max Kaggle 12h usage)
# =================================================================

!pip install -q monai nibabel scikit-learn einops

import os
import logging
import warnings
import torch
import numpy as np
import nibabel as nib
import nibabel.processing
from collections import defaultdict
from sklearn.model_selection import train_test_split 
from tqdm.auto import tqdm

# Suppress Kaggle warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  
os.environ['CUDA_MODULE_LOADING'] = 'LAZY' 
logging.getLogger('absl').setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Importing MONAI components
from monai.networks.nets import DynUNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.transforms import (
    Compose, NormalizeIntensityd, RandCropByPosNegLabeld, 
    RandFlipd, RandRotate90d, CastToTyped, EnsureTyped, SpatialPadd
)
from monai.inferers import sliding_window_inference
from monai.data import decollate_batch, list_data_collate

# --- 1. KAGGLE PATHS & CONFIGURATION ---
CONFIG = {
    "SEARCH_ROOT": "/kaggle/input/datasets/prosenjitmondol/a-stroke-lesion-segmentation-dataset/ISLES-2022",
    "SAVE_DIR": "/kaggle/working/",
    
    "model_name": "DynUNet_P100_Final", 
    "roi_size": (64, 64, 64),
    "batch_size": 2,          
    "accumulation_steps": 2,  
    "epochs": 100,            
    "lr": 2e-4,               
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "seed": 42,
    "split": {"train": 0.70, "val": 0.15, "test": 0.15}
}

os.makedirs(CONFIG["SAVE_DIR"], exist_ok=True)
print(f"🚀 Initializing {CONFIG['model_name']} Engine...")
print(f"⚡ Hardware Fix: 3D Dropout completely disabled (dropout=None)")
print(f"⚡ Pipeline: Crash-Proof Foreground Oversampling ACTIVATED")

# --- 2. DATA PROCESSING ---
def prepare_isles_data(root):
    subjects = defaultdict(dict)
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.endswith(('.nii', '.nii.gz')):
                full_path = os.path.join(dirpath, f)
                sub_id = next((p for p in full_path.split(os.sep) if 'sub-' in p.lower()), os.path.basename(dirpath))
                f_l = f.lower()
                if 'dwi' in f_l: subjects[sub_id]['dwi'] = full_path
                elif 'adc' in f_l: subjects[sub_id]['adc'] = full_path
                elif 'flair' in f_l: subjects[sub_id]['flair'] = full_path
                elif any(x in f_l for x in ['msk', 'mask', 'lesion']): subjects[sub_id]['msk'] = full_path
    
    data = [f for s, f in subjects.items() if all(k in f for k in ['dwi', 'adc', 'flair', 'msk'])]
    data = sorted(data, key=lambda x: list(x.values())[0])  
    return data

class ISLESDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data, self.transform = data, transform
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        p = self.data[idx]
        dwi = nib.load(p['dwi'])
        adc_r = nib.processing.resample_from_to(nib.load(p['adc']), dwi, order=1)
        flr_r = nib.processing.resample_from_to(nib.load(p['flair']), dwi, order=1)
        msk_r = nib.processing.resample_from_to(nib.load(p['msk']), dwi, order=0)

        img = np.stack([np.nan_to_num(dwi.get_fdata()), np.nan_to_num(adc_r.get_fdata()), np.nan_to_num(flr_r.get_fdata())], 0)
        lbl = np.expand_dims(np.nan_to_num(msk_r.get_fdata()), 0)

        del dwi, adc_r, flr_r, msk_r
        d = {"image": img.astype(np.float32), "label": lbl.astype(np.float32)}
        return self.transform(d) if self.transform else d

# --- 3. CRASH-PROOF COLLATE FUNCTION ---
def crash_proof_collate(batch):
    flat_batch = []
    for item in batch:
        if isinstance(item, list):
            flat_batch.extend(item)
        else:
            flat_batch.append(item)
    return list_data_collate(flat_batch)

# --- 4. THE SOTA AUGMENTATION PIPELINE ---
xforms = Compose([
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    SpatialPadd(keys=["image", "label"], spatial_size=CONFIG["roi_size"]),
    
    # 🌟 FOREGROUND OVERSAMPLING: Forces 66% of crops to center perfectly on the stroke lesion
    RandCropByPosNegLabeld(
        keys=["image", "label"], 
        label_key="label", 
        spatial_size=CONFIG["roi_size"], 
        pos=2,      
        neg=1,      
        num_samples=1
    ),
    
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=[0, 1, 2]),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    CastToTyped(keys=["image", "label"], dtype=[torch.float32, torch.float32]),
    EnsureTyped(keys=["image", "label"]),
])

test_transforms = Compose([
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    CastToTyped(keys=["image"], dtype=[torch.float32]),
    EnsureTyped(keys=["image"]),
])

# --- 5. TRAIN / VAL / TEST SPLIT (70/15/15) ---
def split_data(data, seed=42):
    train_ratio = CONFIG["split"]["train"]
    val_ratio = CONFIG["split"]["val"]
    test_ratio = CONFIG["split"]["test"]
    train_data, temp_data = train_test_split(data, train_size=train_ratio, random_state=seed, shuffle=True)
    val_size = int(round(val_ratio / (val_ratio + test_ratio) * len(temp_data)))
    return train_data, temp_data[:val_size], temp_data[val_size:]

# --- 6. THE P100 DynUNet TRAINING ENGINE ---
def run():
    torch.manual_seed(CONFIG["seed"])
    np.random.seed(CONFIG["seed"])
    
    data = prepare_isles_data(CONFIG["SEARCH_ROOT"])
    if len(data) == 0:
        print("❌ No data found.")
        return

    train_data, val_data, test_data = split_data(data, seed=CONFIG["seed"])

    t_ldr = DataLoader(ISLESDataset(train_data, xforms), batch_size=CONFIG["batch_size"], shuffle=True, num_workers=0, collate_fn=crash_proof_collate)
    v_ldr = DataLoader(ISLESDataset(val_data, test_transforms), batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
    test_ldr = DataLoader(ISLESDataset(test_data, test_transforms), batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)

    loss_fn = DiceCELoss(include_background=False, sigmoid=True, squared_pred=True)
    metric = DiceMetric(include_background=False, reduction="mean")
    
    # 🧠 DEPLOYING DynUNet (nnU-Net Architecture) 🧠
    m = DynUNet(
        spatial_dims=3, 
        in_channels=3, 
        out_channels=1,
        kernel_size=[[3,3,3], [3,3,3], [3,3,3], [3,3,3], [3,3,3]], 
        strides=[[1,1,1], [2,2,2], [2,2,2], [2,2,2], [2,2,2]],
        upsample_kernel_size=[[2,2,2], [2,2,2], [2,2,2], [2,2,2]], 
        filters=[16, 32, 64, 128, 256],
        dropout=None, # 🌟 THE HARDWARE CRASH FIX: Completely bypasses the missing CUDA Dropout3d Kernel
        res_block=True
    ).to(CONFIG["device"])

    opt = optim.AdamW(m.parameters(), lr=CONFIG["lr"], weight_decay=1e-5)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG["epochs"])

    best_val = 0.0
    best_model_path = os.path.join(CONFIG["SAVE_DIR"], f"{CONFIG['model_name']}_best.pth")
    accum_steps = CONFIG["accumulation_steps"]

    for ep in range(CONFIG["epochs"]):
        print(f"\nEpoch {ep+1:03d}/{CONFIG['epochs']}")
        m.train()
        l_sum, train_steps = 0.0, 0
        opt.zero_grad()
        
        for b in tqdm(t_ldr, desc="Train", leave=False):
            img, msk = b["image"].to(CONFIG["device"]), b["label"].to(CONFIG["device"])
            train_steps += 1
            
            # Pure FP32 Execution for P100 Stability
            out = m(img)
            loss = loss_fn(out, msk) / accum_steps
            loss.backward()
            
            if train_steps % accum_steps == 0 or train_steps == len(t_ldr):
                torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=2.0)
                opt.step()
                opt.zero_grad()
                
            l_sum += (loss.item() * accum_steps)

        avg_loss = l_sum / train_steps if train_steps > 0 else 0.0
        sch.step()

        # Validation
        m.eval()
        metric.reset()
        with torch.no_grad():
            for vb in v_ldr:
                vi, vm = vb["image"].to(CONFIG["device"]), vb["label"].to(CONFIG["device"])
                vo = sliding_window_inference(vi, CONFIG["roi_size"], sw_batch_size=4, predictor=m, overlap=0.6)
                preds = [torch.sigmoid(i) > 0.5 for i in decollate_batch(vo)]
                metric(y_pred=preds, y=vm)

        cur_val = metric.aggregate().item() if len(val_data) > 0 else 0.0
        print(f"Loss: {avg_loss:.4f} | Val Dice: {cur_val:.4f}")

        if cur_val > best_val:
            best_val = cur_val
            torch.save(m.state_dict(), best_model_path)
            print(f"🌟 New best validation Dice: {best_val:.4f} -> saved")

        torch.cuda.empty_cache()

    # --- 7. FINAL EVALUATION WITH TEST-TIME AUGMENTATION (TTA) ---
    print("\n" + "="*50)
    print("🧠 ACTIVATING TEST-TIME AUGMENTATION (TTA) FOR FINAL SCORES 🧠")
    print("="*50)
    
    if os.path.exists(best_model_path):
        m.load_state_dict(torch.load(best_model_path, map_location=CONFIG["device"]))

    if len(test_data) > 0:
        m.eval()
        metric.reset()
        with torch.no_grad():
            for tb in tqdm(test_ldr, desc="Test Eval (TTA)", leave=False):
                ti, tm = tb["image"].to(CONFIG["device"]), tb["label"].to(CONFIG["device"])
                
                p1 = torch.sigmoid(sliding_window_inference(ti, CONFIG["roi_size"], 4, m, overlap=0.6))
                
                ti_flip_x = torch.flip(ti, dims=[2])
                p2_raw = torch.sigmoid(sliding_window_inference(ti_flip_x, CONFIG["roi_size"], 4, m, overlap=0.6))
                p2 = torch.flip(p2_raw, dims=[2])
                
                ti_flip_y = torch.flip(ti, dims=[3])
                p3_raw = torch.sigmoid(sliding_window_inference(ti_flip_y, CONFIG["roi_size"], 4, m, overlap=0.6))
                p3 = torch.flip(p3_raw, dims=[3])
                
                ensemble_preds = (p1 + p2 + p3) / 3.0
                
                final_preds = [i > 0.5 for i in decollate_batch(ensemble_preds)]
                metric(y_pred=final_preds, y=tm)
                
        print(f"\n🎯 FINAL TEST Dice (F1) WITH nnU-Net & TTA: {metric.aggregate().item():.4f}")

if __name__ == "__main__":
    run()